# SQLite EOD price database — health check

Ad-hoc notebook to inspect `data/sqlite/eod_prices.sqlite`:

- File size, schema, row counts per ticker
- Stored date ranges and data sources
- Coverage vs portfolio holding windows
- Missing / stale / failed tickers

Run from the repo root or from `notebooks/` (path setup handles both).

In [ ]:
import sqlite3
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
if not (ROOT / "degiro_analytics").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from degiro_analytics.data_quality import (
    build_sqlite_completeness_report,
    load_raw_prices_from_db,
    load_sqlite_inventory,
)
from degiro_analytics.degiro_client import extract_traded_isins, fetch_degiro_reports
from degiro_analytics.isin_mapping import build_isin_ticker_map, tickers_for_isins
from degiro_analytics.market_data import _ensure_price_db
from degiro_analytics.portfolio_core import build_positions_from_trades, build_trades_table
from degiro_analytics.settings import (
    CACHE_DIR,
    DEFAULT_BENCHMARKS,
    EXPORT_DIR,
    FULL_HISTORY_START,
    PRICE_DB_PATH,
)

pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [ ]:
# --- configuration ---
DB_PATH = PRICE_DB_PATH
ANALYSIS_START = FULL_HISTORY_START          # e.g. "1990-01-01"
ANALYSIS_END = pd.Timestamp.today().strftime("%Y-%m-%d")
USE_DEGIRO_CACHE = True                      # set False only if you want live API fetch
REFRESH_PRICES = False                       # True = download missing EOD before report

print(f"SQLite path: {DB_PATH}")
print(f"Analysis window: {ANALYSIS_START} .. {ANALYSIS_END}")

## 1. Raw database inspection

In [ ]:
def inspect_sqlite(db_path: Path) -> dict:
    db_path = Path(db_path)
    info = {
        "exists": db_path.exists(),
        "path": str(db_path.resolve()),
        "size_mb": round(db_path.stat().st_size / 1_048_576, 2) if db_path.exists() else 0,
    }
    if not db_path.exists():
        return info

    conn = sqlite3.connect(db_path)
    tables = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn
    )
    info["tables"] = tables["name"].tolist()

    schema_rows = []
    for table in info["tables"]:
        cols = pd.read_sql_query(f"PRAGMA table_info({table})", conn)
        for _, row in cols.iterrows():
            schema_rows.append({"table": table, "column": row["name"], "type": row["type"]})
    info["schema"] = pd.DataFrame(schema_rows)

    info["eod_prices_rows"] = pd.read_sql_query("SELECT COUNT(*) AS n FROM eod_prices", conn).iloc[0, 0]
    info["eod_meta_rows"] = pd.read_sql_query("SELECT COUNT(*) AS n FROM eod_meta", conn).iloc[0, 0]
    info["date_range"] = pd.read_sql_query(
        "SELECT MIN(date) AS first_date, MAX(date) AS last_date FROM eod_prices", conn
    )
    info["sources"] = pd.read_sql_query(
        "SELECT source, COUNT(*) AS rows FROM eod_prices GROUP BY source ORDER BY rows DESC", conn
    )
    info["top_tickers_by_rows"] = pd.read_sql_query(
        "SELECT ticker, COUNT(*) AS rows, MIN(date) AS first_date, MAX(date) AS last_date "
        "FROM eod_prices GROUP BY ticker ORDER BY rows DESC LIMIT 15",
        conn,
    )
    conn.close()
    return info


db_info = inspect_sqlite(DB_PATH)

if not db_info["exists"]:
    display(Markdown(f"**Database not found** at `{DB_PATH}`. Run `scripts/update_eod_prices.py` first."))
else:
    display(Markdown(
        f"""### Database overview
- **Path:** `{db_info['path']}`
- **Size:** {db_info['size_mb']} MB
- **Tables:** {', '.join(db_info['tables'])}
- **Price rows:** {db_info['eod_prices_rows']:,}
- **Meta rows:** {db_info['eod_meta_rows']:,}
- **Global date range:** {db_info['date_range'].iloc[0]['first_date']} .. {db_info['date_range'].iloc[0]['last_date']}
"""
    ))
    display(db_info["schema"])
    display(db_info["sources"])
    display(db_info["top_tickers_by_rows"])

## 2. Inventory (`eod_meta` + row counts)

In [ ]:
inventory = load_sqlite_inventory(DB_PATH)
print(f"Tickers in SQLite: {len(inventory)}")
if inventory.empty:
    display(Markdown("No metadata yet — database is empty or missing `eod_meta`."))
else:
    display(
        inventory.assign(
            span_days=(
                pd.to_datetime(inventory["last_date"]) - pd.to_datetime(inventory["first_date"])
            ).dt.days
        ).sort_values(["last_date", "ticker"], ascending=[False, True])
    )

## 3. Expected tickers (portfolio + benchmarks)

Uses cached DeGiro reports to derive the ticker universe — no live API call unless cache is missing.

In [ ]:
reports = fetch_degiro_reports(cache_dir=CACHE_DIR, use_cache=USE_DEGIRO_CACHE)
account = reports["account"]
positions_report = reports["positions"]

traded_universe = extract_traded_isins(account, positions_report)
traded_isins = traded_universe["ISIN"].tolist()
isin_map, isin_mapping_report = build_isin_ticker_map(traded_isins)
portfolio_tickers = tickers_for_isins(traded_isins, isin_map)
benchmark_tickers = list(DEFAULT_BENCHMARKS.values())
expected_tickers = sorted(set(portfolio_tickers) | set(benchmark_tickers))

trades = build_trades_table(account, isin_map)
positions = build_positions_from_trades(trades)

in_db = set(inventory["ticker"]) if not inventory.empty else set()
missing_from_db = sorted(set(expected_tickers) - in_db)
extra_in_db = sorted(in_db - set(expected_tickers))

print(f"Portfolio tickers: {len(portfolio_tickers)}")
print(f"Benchmark tickers: {len(benchmark_tickers)}")
print(f"Expected total:    {len(expected_tickers)}")
print(f"In SQLite:         {len(in_db)}")
print(f"Missing from DB:   {len(missing_from_db)}")
print(f"Extra in DB:       {len(extra_in_db)}")

if missing_from_db:
    display(Markdown("**Expected but not stored:**"))
    display(pd.DataFrame({"ticker": missing_from_db}))
if extra_in_db:
    display(Markdown("**Stored but not in current universe (legacy / benchmarks):**"))
    display(pd.DataFrame({"ticker": extra_in_db}))

## 4. Completeness report

Compares available business-day prices during each ticker's holding window (or full analysis window for benchmarks).

In [ ]:
db_update = pd.DataFrame(columns=["ticker", "status", "error"])

if REFRESH_PRICES:
    from degiro_analytics.market_data import update_local_price_db

    prices, db_update = update_local_price_db(
        expected_tickers,
        start_date=ANALYSIS_START,
        end_date=ANALYSIS_END,
        db_path=DB_PATH,
        full_history_start=ANALYSIS_START,
    )
    inventory = load_sqlite_inventory(DB_PATH)

ticker_roles = {t: "portfolio" for t in portfolio_tickers}
ticker_roles.update({t: "benchmark" for t in benchmark_tickers})

sqlite_report = build_sqlite_completeness_report(
    expected_tickers=expected_tickers,
    positions=positions,
    analysis_start=ANALYSIS_START,
    analysis_end=ANALYSIS_END,
    db_path=DB_PATH,
    db_update=db_update,
    inventory=inventory,
    ticker_roles=ticker_roles,
)

summary = sqlite_report["summary"]
by_ticker = sqlite_report["by_ticker"]
issues = sqlite_report["issues"]

display(Markdown("### Summary"))
display(summary.to_frame("value"))
print("\n" + sqlite_report["report_text"])

In [ ]:
status_order = ["ok", "stale", "partial_coverage", "critical_gaps", "missing_from_sqlite"]
by_ticker["status"] = pd.Categorical(by_ticker["status"], categories=status_order, ordered=True)

display(Markdown("### Per-ticker detail (sorted by status)"))
display(
    by_ticker[
        [
            "ticker", "role", "status", "coverage_ratio", "expected_points",
            "available_points", "missing_points", "db_first_date", "db_last_date",
            "db_source", "stale_days_vs_analysis_end", "update_status", "update_error",
        ]
    ].sort_values(["status", "coverage_ratio"])
)

if not issues.empty:
    display(Markdown(f"### Issues ({len(issues)} tickers)"))
    display(issues[["ticker", "role", "status", "coverage_ratio", "db_last_date", "update_error"]])

## 5. Charts

In [ ]:
color_map = {
    "ok": "#2ca02c",
    "stale": "#ff7f0e",
    "partial_coverage": "#ffbb78",
    "critical_gaps": "#d62728",
    "missing_from_sqlite": "#9467bd",
}

if by_ticker.empty:
    print("No tickers to chart.")
else:
    fig_cov = px.bar(
        by_ticker.sort_values("coverage_ratio"),
        x="ticker",
        y="coverage_ratio",
        color="status",
        color_discrete_map=color_map,
        title="Price coverage during holding / analysis window",
        labels={"coverage_ratio": "Coverage ratio"},
    )
    fig_cov.update_layout(template="plotly_white", xaxis_tickangle=-45)
    fig_cov.update_yaxes(tickformat=".0%")
    fig_cov.show()

    status_counts = by_ticker["status"].value_counts().reset_index()
    status_counts.columns = ["status", "count"]
    fig_status = px.bar(
        status_counts,
        x="status",
        y="count",
        color="status",
        color_discrete_map=color_map,
        title="Tickers by health status",
    )
    fig_status.update_layout(template="plotly_white", showlegend=False)
    fig_status.show()

    if not inventory.empty:
        inv_plot = inventory.copy()
        inv_plot["first_date"] = pd.to_datetime(inv_plot["first_date"])
        inv_plot["last_date"] = pd.to_datetime(inv_plot["last_date"])
        fig_span = px.scatter(
            inv_plot,
            x="first_date",
            y="last_date",
            size="row_count",
            hover_data=["ticker", "source", "row_count"],
            title="Stored date span per ticker (bubble size = row count)",
        )
        fig_span.update_layout(template="plotly_white")
        fig_span.show()

## 6. Drill-down: missing dates for one ticker

Pick a ticker with gaps to see which business days have no price in SQLite.

In [ ]:
DRILL_TICKER = issues.iloc[0]["ticker"] if not issues.empty else (
    expected_tickers[0] if expected_tickers else None
)
print(f"Drill-down ticker: {DRILL_TICKER}")

if DRILL_TICKER:
    row = by_ticker.loc[by_ticker["ticker"] == DRILL_TICKER].iloc[0]
    window_start = pd.to_datetime(row["holding_start"])
    window_end = pd.to_datetime(row["holding_end"])
    window_index = pd.bdate_range(window_start, window_end)

    prices = load_raw_prices_from_db(
        [DRILL_TICKER],
        window_start.strftime("%Y-%m-%d"),
        window_end.strftime("%Y-%m-%d"),
        DB_PATH,
    )
    series = prices[DRILL_TICKER] if DRILL_TICKER in prices.columns else pd.Series(dtype=float)
    aligned = series.reindex(window_index)
    missing_dates = aligned[aligned.isna()].index

    display(Markdown(
        f"""**{DRILL_TICKER}** | status={row['status']} | coverage={row['coverage_ratio']:.1%}
- Window: {window_start.date()} .. {window_end.date()}
- DB range: {row['db_first_date']} .. {row['db_last_date']} ({row['db_source']})
- Missing business days: {len(missing_dates)} / {len(window_index)}
"""
    ))

    fig_price = go.Figure()
    fig_price.add_trace(go.Scatter(x=aligned.index, y=aligned.values, mode="lines", name="close"))
    fig_price.update_layout(
        title=f"{DRILL_TICKER} — prices in SQLite",
        template="plotly_white",
        xaxis_title="Date",
        yaxis_title="Close",
    )
    fig_price.show()

    if len(missing_dates):
        display(Markdown("First 30 missing dates:"))
        display(pd.DataFrame({"missing_date": missing_dates[:30]}))

## 7. Export snapshot (optional)

Writes CSV copies under `data/exports/` for diffing across runs.

In [ ]:
EXPORT_SNAPSHOT = True

if EXPORT_SNAPSHOT:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    inventory.to_csv(EXPORT_DIR / "sqlite_inventory_snapshot.csv", index=False)
    by_ticker.to_csv(EXPORT_DIR / "sqlite_completeness_snapshot.csv", index=False)
    issues.to_csv(EXPORT_DIR / "sqlite_issues_snapshot.csv", index=False)
    summary.to_frame("value").to_csv(EXPORT_DIR / "sqlite_summary_snapshot.csv")
    print(f"Exported to {EXPORT_DIR.resolve()}")